## PX1 Mini-Cam — восьмой этап: идентификация реальных программ и модулей

Найденные исполняемые файлы `/root/ProteusApp` и `/root/UDProteus` — ARM ELF. Этот блокнот **не запускает программы из образа** и не проверяет сети Mini-Cam. Он проверяет форматы, хэши SHA-256, Build-ID, имена зависимостей, аппаратные модули и **только названия** упоминаемых программ в сценариях загрузки. Код и сырые двоичные файлы не публикуются. Отчёт записывается только в твой Google Диск.

Запуск на Android: **«Среда выполнения» → «Выполнить все»**. Ожидаемый файл: `PX1_SD_BINARY_IDENTIFIERS_REPORT.txt`. Временный раздел (~7.9 ГБ) извлекается на сервер Colab и удаляется. Исходный файл `ЛП.imgc` не изменяется.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, struct, shutil
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
src = root / 'Untitled folder' / 'ЛП.imgc'
if not src.is_file():
    src = next((Path(d)/'ЛП.imgc' for d,_,names in os.walk(root) if 'ЛП.imgc' in names), None)
if not src or not src.is_file(): raise FileNotFoundError('Не найден ЛП.imgc')
project=Path('/content/unimgc')
if not project.is_dir():
    subprocess.run(['git','clone','--depth','1','https://github.com/Shizmob/unimgc.git',str(project)],check=True)
subprocess.run(['make','-C',str(project)],check=True)
part=Path('/content/px1_ro_partition.img')
offset=32901120
size=7912719360
if shutil.disk_usage('/content').free < size + 1024**3:
    raise RuntimeError('Мало свободного места в Colab для копии раздела')
print('Читаю раздел в временный файл Colab (примерно 7.9 ГБ)...')
p=subprocess.Popen([str(project/'unimgc'),str(src)],stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
try:
    toskip=offset
    while toskip:
        data=p.stdout.read(min(4*1024*1024,toskip))
        if not data: raise EOFError('Образ закончился до начала раздела')
        toskip-=len(data)
    left=size
    with part.open('wb') as out:
        while left:
            data=p.stdout.read(min(4*1024*1024,left))
            if not data: raise EOFError('Образ закончился внутри раздела')
            out.write(data)
            left-=len(data)
finally:
    p.stdout.close()
    p.terminate()
    try: p.wait(timeout=10)
    except subprocess.TimeoutExpired: p.kill(); p.wait()
def dbg(command):
    v=subprocess.run(['debugfs','-R',command,str(part)],capture_output=True,timeout=90)
    return v.stdout.decode('utf-8','replace')
# Only local owner-controlled image data is read. No bundled software is executed.
import re, hashlib
report=[
    'PX1 Mini-Cam / этап 8: идентификаторы программ и модулей (offline, read-only)',
    'Source: owner-controlled SD image, no secrets or customer data in output'
]
def line(label, value):
    report.append(label + ': ' + str(value))
def dump_file(image_path, local_path):
    local_path.unlink(missing_ok=True)
    dbg('dump '+image_path+' '+str(local_path))
    return local_path.is_file()
filetypes=['/root/ProteusApp','/root/UDProteus','/root/Proteus',
           '/root/ProteusFPHub_ext.ko','/root/rp_enc.ko']
known_names=[b'ProteusApp',b'UDProteus',b'/dev/tty',b'/dev/video',
             b'/dev/i2c',b'/dev/spi',b'/dev/fb',b'/dev/input',
             b'gstreamer',b'VPU',b'IPU',b'RS485',b'Serial']
for pos, f in enumerate(filetypes):
    temp=Path('/content/px1_target_'+str(pos))
    if not dump_file(f,temp):
        line('Объект '+f,'не найден или не извлечён');continue
    h=hashlib.sha256()
    with temp.open('rb') as fh:
        while True:
            buf=fh.read(2**20)
            if not buf:break
            h.update(buf)
    line('Имя',f)
    line('Размер байт',temp.stat().st_size)
    line('SHA256 (идентификация версии, не проверка уязвимости)',h.hexdigest())
    first=temp.open('rb').read(512)
    if first.startswith(b'\x7fELF'):
        rd=subprocess.run(['readelf','-h',str(temp)],capture_output=True,text=True,timeout=60)
        for row in rd.stdout.splitlines():
            if row.lstrip().startswith(('Class:','Type:','Machine:')):
                report.append('  ELF '+row.strip())
        notes=subprocess.run(['readelf','-n',str(temp)],capture_output=True,text=True,timeout=60)
        builds=re.findall(r'Build ID:\s*([0-9a-fA-F]+)',notes.stdout)
        if builds:line('ELF Build ID',builds[0])
        dyn=subprocess.run(['readelf','-d',str(temp)],capture_output=True,text=True,timeout=60)
        needed=sorted(set(re.findall(r'Shared library: \[([^\]]+)\]',dyn.stdout)))
        line('Динамические библиотеки',', '.join(needed) if needed else 'не определены')
        if f.endswith('.ko') and shutil.which('modinfo'):
            proc2=subprocess.run(['modinfo','-F','vermagic',str(temp)],capture_output=True,text=True,timeout=60)
            ver=proc2.stdout.strip()
            if ver:line('Module vermagic',ver[:200])
        if f.endswith(('ProteusApp','UDProteus')):
            with temp.open('rb') as fh: binary=fh.read()
            found=[name.decode('ascii') for name in known_names if name.lower() in binary.lower()]
            line('Только метки возможных интерфейсов (не доказательство использования)',', '.join(found) or 'нет')
    else:
        desc=subprocess.run(['file','-b',str(temp)],capture_output=True,text=True,timeout=60)
        line('Формат',desc.stdout.strip()[:160])
        if first.startswith(b'#!'):
            line('Тип','возможный сценарий с shebang')
        elif first.startswith(b'PK\x03\x04'):
            line('Тип','ZIP')
        elif first.startswith(b'<?xml'):
            line('Тип','XML')
    temp.unlink(missing_ok=True)

# Scripts inspected only for command NAMES. No contents are published.
for script in ['/root/Proteus','/root/startup.sh','/root/RunOnceScript.sh',
               '/etc/rc.d/rc.local','/etc/rc.d/rcS']:
    raw=dbg('cat '+script)
    if len(raw)>160000:continue
    markers=['ProteusApp','UDProteus','Proteus','startSamba','dropbear',
             'smbd','boa','wpa_supplicant','telnetd','Firmware']
    active=[a for a in raw.splitlines() if a.strip() and not a.lstrip().startswith('#')]
    found=[k for k in markers if any(re.search(r'(?<![A-Za-z_])'+re.escape(k)+r'(?![A-Za-z_])',a,re.I) for a in active)]
    line('Сценарий '+script+' — упомянутые имена',', '.join(found) if found else 'не найдены')
    line('Сценарий '+script+' — число активных строк',len(active))

report.append('Никаких CVE или доступных сетевых служб автоматически не подтверждено.')
report_file=root/'PX1_SD_BINARY_IDENTIFIERS_REPORT.txt'
report_file.write_text('\n'.join(report)+'\n',encoding='utf-8')
part.unlink(missing_ok=True)
print('\n'.join(report))
print('ГОТОВО. Приватный отчёт:',report_file.name)
